# CSCI E-89 Deep Learning — Assignment 03

**Name:** Jazmyn Stokes
**Problem 2 (5%):** Wrap the Problem 1 process into a single, self-contained Jupyter
notebook — the same PyTorch Fashion MNIST image classifier (with the required
training-accuracy plot), merged from the nine generated scripts into one notebook
that reads start to finish, with imports consolidated at the top instead of
scattered across sections, and one markdown cell per numbered section explaining it.
Cells all run top to bottom in one shared namespace.


---
## 0. Imports

All imports used anywhere in this notebook, gathered in one place instead of
scattered per section — the nine individual scripts each imported what they needed
as they went (e.g. `torchvision` only in the data-loading script); here everything
is pulled to the top so the notebook reads as one continuous program.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader


---
## 1. Setup — device, seed, plotting defaults

Pick the best available device (CUDA, then Apple Silicon's `mps`, then CPU), fix the random seed so the split/weights/shuffling are reproducible, set matplotlib defaults used by every later figure, and print the PyTorch version and selected device.


In [ ]:
# Device selection, reproducibility seed, and plot defaults.
# Pick the fastest device available on this machine.
# Order matters: CUDA (NVIDIA GPU) first, then Apple Silicon's "mps" backend
# (what this Mac uses), then CPU as the universal fallback so the notebook still
# runs anywhere.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# Fix the random seed once, up front, so the dataset split, weight initialization,
# and DataLoader shuffling are all reproducible on every rerun of this notebook.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Matplotlib defaults applied notebook-wide, so every later figure (e.g. the
# training/validation accuracy curves) is consistently sized and legible without
# repeating these settings near each plot.
plt.rc("font", size=12)
plt.rc("axes", labelsize=12, titlesize=14)
plt.rc("legend", fontsize=12)
plt.rc("figure", figsize=(8, 5))

print(f"PyTorch version : {torch.__version__}")
print(f"Selected device : {device}")


---
## 2. Load Fashion MNIST and split into train / validation

Download (or reuse the local cache of) Fashion MNIST, scale pixels to [0, 1] with `transforms.v2`, re-seed immediately before splitting so the partition is reproducible regardless of what ran earlier, and split the 60,000 training images into 55,000 train / 5,000 validation. The 10,000-image test set is loaded here but not touched again until Section 9.


In [ ]:
# Step 2 — Load Fashion MNIST, scale to [0, 1], re-seed, split 55k/5k train/validation.
# Depends on `torch`, `SEED` from the setup cell above.


# ToImage() gives a tensor in [C, H, W] layout; ToDtype(..., scale=True) casts the
# uint8 pixels to float32 and divides by 255, putting every input in [0, 1].
to_float_tensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=to_float_tensor)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=to_float_tensor)

# Human-readable label names, in class-index order (0-9).
class_names = train_and_valid_data.classes

# Re-seed right before the split (as requested) so the 55k/5k partition is
# reproducible regardless of what ran earlier in the session.
torch.manual_seed(SEED)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55000, 5000])

print(f"train      : {len(train_data):,} images")
print(f"validation : {len(valid_data):,} images")
print(f"test       : {len(test_data):,} images")
print(f"class names: {class_names}")


---
## 3. DataLoaders

Wrap the three splits in `DataLoader`s (`batch_size=32`, shuffling only the training loader), then print one raw sample's shape/dtype/label as a quick check before any model sees the data.


In [ ]:
# Step 3 — DataLoaders for train/validation/test, then inspect one sample.
# Depends on `train_data`, `valid_data`, `test_data`, `class_names`, `SEED` from
# Step 2.


torch.manual_seed(SEED)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

# Each dataset entry is an (image, label) tuple; look at the very first one.
X_sample, y_sample = train_data[0]
print("sample image shape :", tuple(X_sample.shape))   # [channels, rows, cols]
print("sample image dtype :", X_sample.dtype)
print("sample label       :", y_sample, "->", class_names[y_sample])


---
## 4. Model and loss

`ImageClassifier`: `Flatten -> Linear(784,300) -> ReLU -> Linear(300,100) -> ReLU -> Linear(100,10)`, with no activation on the output layer since `nn.CrossEntropyLoss` expects raw logits and applies log-softmax internally. The model is moved to `device`, the loss is created, and the model plus its parameter count are printed.


In [ ]:
# Step 4 — Define the ImageClassifier MLP, move it to device, create the loss.
# Depends on `nn`, `device` from Step 1.

class ImageClassifier(nn.Module):
    """Fully connected classifier for 28x28 grayscale Fashion MNIST images.

    Architecture: Flatten -> Linear(784,300) -> ReLU -> Linear(300,100) -> ReLU
    -> Linear(100,10). The final layer outputs raw logits (no activation), since
    nn.CrossEntropyLoss expects logits and applies log-softmax internally.
    """

    def __init__(self, n_inputs=784, n_hidden1=300, n_hidden2=100, n_classes=10):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),                          # [N, 1, 28, 28] -> [N, 784]
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),        # raw logits, no activation
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(SEED)  # reproducible weight initialization
model = ImageClassifier(784, 300, 100, 10).to(device)

# Standard multi-class classification loss: expects raw logits + integer class
# labels, and applies log-softmax internally.
criterion = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\ntotal trainable parameters: {n_params:,}")


---
## 5. Training and evaluation functions

`evaluate_tm()` runs a loader under `no_grad` and returns the aggregated metric; `train2()` runs the training loop for `n_epochs`, printing loss/accuracy each epoch and returning a `history` dict (`train_losses`, `train_metrics`, `valid_metrics`) so the next section can plot it. Only defined here — nothing is executed yet.


In [ ]:
# Step 5 — Training/evaluation helper functions only. Nothing is executed here;
# no optimizer or training run exists yet, so this cell just defines the two
# functions used by the next step.

def evaluate_tm(model, data_loader, metric):
    """Run `model` over every batch in `data_loader` under no_grad and return the
    aggregated value of `metric` (a torchmetrics metric object)."""
    model.eval()           # eval mode: disables dropout / uses running BN stats
    metric.reset()         # torchmetrics accumulates internally; clear stale state
    with torch.no_grad():  # no autograd graph needed for evaluation
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train2(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    """Train `model` for `n_epochs`, printing loss/accuracy each epoch, and return
    a history dict with keys 'train_losses', 'train_metrics', 'valid_metrics'
    (one entry per epoch) so the caller can plot learning curves afterward."""
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}

    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        running_loss, n_seen = 0.0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
            n_seen += X_batch.size(0)
            metric.update(y_pred, y_batch)

        train_loss = running_loss / n_seen
        train_metric = metric.compute().item()
        valid_metric = evaluate_tm(model, valid_loader, metric).item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(valid_metric)

        print(f"epoch {epoch + 1:2d}/{n_epochs} | "
              f"train loss: {train_loss:.4f} | "
              f"train acc: {train_metric:.4f} | "
              f"valid acc: {valid_metric:.4f}")

    return history


print("evaluate_tm() and train2() defined. Nothing executed yet.")


---
## 6. Run training — 20 epochs, SGD(lr=0.1)

The actual training run: plain SGD at `lr=0.1`, a `torchmetrics` multiclass accuracy on the same device, 20 epochs, keeping the returned `history`. This is the one long-running cell in the notebook.


In [ ]:
# Step 6 — Train for 20 epochs with SGD(lr=0.1); keep the returned history.
# Depends on `model`, `criterion`, `device` (Step 4), `train_loader`, `valid_loader`
# (Step 3), and `train2`/`evaluate_tm` (Step 5). This is the long-running cell —
# expect it to take a few minutes.

n_epochs = 20

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# The metric must live on the same device as the tensors it consumes, otherwise
# torchmetrics raises a device-mismatch error.
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

history = train2(model, optimizer, criterion, accuracy,
                  train_loader, valid_loader, n_epochs)


---
## 7. Plot training accuracy (required) and training loss

Training accuracy is a running average over the epoch, so it's plotted at the epoch midpoint (`epoch + 0.5`); validation accuracy is measured once at the epoch's end (`epoch + 1.0`). Labeled, gridded, legended, y-axis fixed to `[0.7, 1.0]`. Final numbers are printed, and a second figure plots training loss as a cross-check.


In [ ]:
# Step 7 — Plot training vs. validation accuracy (the assignment's required
# addition), print the final numbers, and plot training loss as a cross-check.
# Depends on `history`, `n_epochs` from Step 6.

epochs = np.arange(n_epochs)

# Training accuracy is a running average over the epoch, so it belongs at the
# epoch's midpoint (epoch + 0.5). Validation accuracy is measured once, at the
# end of the epoch, so it sits at epoch + 1.0.
plt.figure()
plt.plot(epochs + 0.5, history["train_metrics"], ".--", label="Training accuracy")
plt.plot(epochs + 1.0, history["valid_metrics"], ".-", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fashion MNIST classifier - learning curves")
plt.grid(True)
plt.legend()
plt.axis([0.5, n_epochs, 0.7, 1.0])
plt.show()

print(f"final training accuracy   : {history['train_metrics'][-1]:.4f}")
print(f"final validation accuracy : {history['valid_metrics'][-1]:.4f}")

# Second figure: training loss per epoch, as a cross-check. Accuracy rising while
# loss falls is the expected pattern for a healthy training run.
plt.figure()
plt.plot(epochs + 0.5, history["train_losses"], ".--", color="tab:red",
         label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Fashion MNIST classifier - training loss")
plt.grid(True)
plt.legend()
plt.show()

print(f"final training loss       : {history['train_losses'][-1]:.4f}")


---
## 8. Predict on 3 validation images and visualize

Predicts on 3 validation images, prints predicted vs. actual class names, softmaxes the logits into probabilities and prints them rounded plus the top-4 per image (moving to `cpu` first on `mps`, since `round()` isn't implemented there), then displays the 3 images with their labels.


In [ ]:
# Step 8 — Predict on 3 validation images, show softmax probabilities + top-4,
# then display the images with their predicted/true labels.
# Depends on `model`, `device`, `valid_loader`, `class_names` from earlier steps.

model.eval()

# Take the first batch from the validation loader and predict on its first 3
# images.
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
y_new = y_new[:3]

with torch.no_grad():
    y_pred_logits = model(X_new)

y_pred = y_pred_logits.argmax(dim=1)  # index of the largest logit per image

print("predicted :", [class_names[i] for i in y_pred])
print("actual    :", [class_names[i] for i in y_new])
print("correct   :", (y_pred.cpu() == y_new).tolist())

# The model outputs logits; softmax turns them into class probabilities.
y_proba = F.softmax(y_pred_logits, dim=1)

# round(decimals=...) is not implemented on the "mps" backend, so move to cpu
# first whenever that's the active device (a no-op cost on cuda/cpu).
if device == "mps":
    y_proba = y_proba.cpu()

print("\nclass probabilities (rounded):")
print(y_proba.round(decimals=3))

# Top-4 classes per image, re-softmaxed over just those 4 logits so the reported
# values sum to 1 per image.
y_top4_values, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_values, dim=1)
if device == "mps":
    y_top4_probas = y_top4_probas.cpu()

print("\ntop-4 probabilities:")
print(y_top4_probas.round(decimals=3))
print("\ntop-4 class indices:")
print(y_top4_indices)


# Display the 3 validation images with their predicted vs. true class names.
fig, axes = plt.subplots(1, 3, figsize=(9, 3.4))
for ax, image, pred, true in zip(axes, X_new.cpu(), y_pred.cpu(), y_new):
    ax.imshow(image.squeeze(), cmap="binary")  # squeeze drops the channel axis
    ax.set_title(f"pred: {class_names[pred]}\ntrue: {class_names[true]}",
                 fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()


---
## 9. Final evaluation on the test set

The only use of `test_loader` anywhere in the notebook — a single, final pass over the 10,000 held-out test images, printing test accuracy and the model's parameter count.


In [ ]:
# Step 9 — Final, single evaluation on the held-out test set.
# Depends on `model`, `test_loader`, `accuracy` from earlier steps. This is the
# only place test_loader is used.

test_accuracy = evaluate_tm(model, test_loader, accuracy).item()
n_params_final = sum(p.numel() for p in model.parameters())

print(f"test accuracy      : {test_accuracy:.4f}")
print(f"model parameters   : {n_params_final:,}")
